# Phát hiện đèn, biển, hàng rào car6: YOLO 9 lớp, so sánh D1–D5
Notebook huấn luyện bộ phát hiện cho xe car6 theo quyết định train v0 ngày 01/11 (`ucr_car6_data_handover.md` mục 5, 6).

- **Lớp (thứ tự cố định, là hợp đồng với C++):** `red`, `green`, `off`, `left`, `right`, `no_left`, `no_right`, `round_barrier`, `work_barrier`. Roboflow xếp lớp theo bảng chữ cái nên nhãn được **ánh xạ theo tên**. Không có lớp người.
- **Chia tập:** theo lượt, đọc `split_v0.csv`; tên Roboflow `..._jpg.rf.<mã>` được đưa về `car6_pNNNN_MMMM`.
- **Tiền xử lý:** letterbox từ ảnh 1280×720: thu nhỏ giữ tỉ lệ bằng `cv2.INTER_AREA`, đệm 114, căn giữa; RGB, chia 255 ngoài mạng. Ảnh train được thu nhỏ trước bằng INTER_AREA để train và suy luận dùng cùng phép thu nhỏ.
- **Tăng cường:** tắt lật ngang ngẫu nhiên và biến đổi hình học của Ultralytics. Tạo sẵn `AUG_COPIES` bản đổi màu/sáng cho mỗi ảnh train và bản lật ngang có đổi nhãn **chỉ cho ảnh chứa `left`/`right`, không chứa `no_left`/`no_right`** (lật biển cấm rẽ làm gạch chéo sai hướng).

| Mã | Mô hình | Cỡ train (cạnh dài) | Cỡ suy luận/xuất (cao×rộng) | Đo trên Nano FP16 |
|---|---|---|---|---|
| D1 | YOLOv8n | 320 | 320×320 | 15,0 ms |
| D2 | YOLOv8n | 416 | 256×416 | chưa đo |
| D3 | YOLOv8n | 416 | 416×416 | 24,5 ms |
| D4 | YOLOv8n | 640 | 384×640 | chưa đo (ước ~35 ms) |
| D5 | YOLO11n | 320 | 320×320 | 15,7 ms; có attention, cần thử TensorRT 8.2 |

D2 và D3 dùng chung một lần train (ảnh vuông 416 có letterbox), chỉ khác cỡ khi suy luận/xuất: letterbox 16:9 vào ô vuông 416 cho cùng tỉ lệ thu nhỏ với ô 256×416 nên không cần `rect=True` lúc train (chế độ này tắt xáo trộn ảnh).

Ba chế độ `MODE`:
1. `"compare"`: mỗi biến thể × mỗi seed: train, đánh giá val ở 5 mức sáng × các ngưỡng conf, ghi thêm dòng vào `Vong3/compare/yolo_compare.csv`. Chạy lại sau khi Colab ngắt: lần chạy đã có trong CSV được bỏ qua, trọng số đã train (trong `Vong3/yolo/weights/<DATA_TAG>/`) được dùng lại.
2. `"final"`: FINAL_FIT trên toàn bộ ảnh có nhãn với `FINAL_VARIANT`, lấy epoch cuối, xuất ONNX.
3. `"prelabel"`: dự đoán ảnh `car6_det` chưa vào Dataset và gửi lên Roboflow dạng dự đoán chờ duyệt.

**Quy trình gán nhãn sơ bộ:** vẽ tay ~30% mỗi lô det → tạo version Roboflow → `MODE="compare"`, `VARIANTS=["D1"]`, `SEEDS=[0]` để có trọng số nhanh → `MODE="prelabel"` với trọng số đó → người sửa trên Roboflow → version mới, đổi `DATA_TAG` → train lại.

Chỉ số val chỉ dùng để lọc thô; quyết định cuối dựa trên FPS đo trên xe và chạy thật. `FAKE_DATA=True` kiểm quy trình trên CPU bằng ảnh giả.

In [ ]:
MODE = "compare"          # "compare" | "final" | "prelabel"
FAKE_DATA = True
DATA_TAG = "det_b1"       # Đổi khi dùng version Roboflow mới
RF_PROJECT = "car6_det"
RF_WORKSPACE = None
RF_VERSION = 1            # Version Roboflow: không Resize, không Augmentation
VARIANTS = ["D1", "D2", "D3", "D5"]   # Thêm "D4" nếu đủ thời gian GPU
SEEDS = [0, 1]
EPOCHS = 60               # T4: ~25-40 phút một lần train ở 320-416
BATCH = 32
AUG_COPIES = 1            # Số bản đổi màu cho mỗi ảnh train (giữ cả ảnh gốc)
FLIP_SIGNS = True         # Tạo bản lật + đổi nhãn left <-> right
CONF_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]
EVAL_CONF = 0.25          # Ngưỡng dự kiến dùng khi thi
MATCH_IOU = 0.5
NMS_IOU = 0.45
BRIGHTNESS = [0.6, 0.8, 1.0, 1.2, 1.4]
EXPORT_FIRST_SEED = True  # Compare: xuất ONNX cho seed đầu của mỗi biến thể
FINAL_VARIANT = "D1"
FINAL_SEED = 0
PRELABEL_WEIGHTS = ""     # File .pt trong Vong3 (cạnh nó phải có .json cùng tên)
PRELABEL_CONF = 0.35
PRELABEL_LIMIT = 20       # None = xử lý hết ảnh đang chờ
SPLIT_CSV = "data/split_v0.csv"
INSTALL_DEPS = True
LOCAL_ROOT = "./MyDrive/UIT_Car_Racing/Vong3"
WORK_DIR = "/content/work_yolo"

CLASS_NAMES = ["red", "green", "off", "left", "right", "no_left", "no_right", "round_barrier", "work_barrier"]
FLIP_SWAP = {"left": "right", "right": "left"}
NO_FLIP = {"no_left", "no_right"}
VARIANT_SPECS = {
    "D1": dict(model="yolov8n", train=320, hw=(320, 320)),
    "D2": dict(model="yolov8n", train=416, hw=(256, 416)),
    "D3": dict(model="yolov8n", train=416, hw=(416, 416)),
    "D4": dict(model="yolov8n", train=640, hw=(384, 640)),
    "D5": dict(model="yolo11n", train=320, hw=(320, 320)),
}

## Môi trường, thư mục và hàm dùng chung
Kết quả ở `MyDrive/UIT_Car_Racing/Vong3/yolo/`; bảng so sánh ở `Vong3/compare/yolo_compare.csv`. Dữ liệu và thư mục train nằm trên đĩa Colab. Khoá Roboflow lấy từ Colab Secrets `ROBOFLOW_API_KEY` hoặc `getpass`.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
IN_COLAB = importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if INSTALL_DEPS:
    pkgs = ["numpy<3", "opencv-python-headless", "onnx>=1.14,<2", "onnxruntime>=1.16,<2", "onnxslim",
            "ultralytics==8.4.161", "roboflow", "requests"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/UIT_Car_Racing/Vong3")
else:
    ROOT = Path(LOCAL_ROOT).expanduser().resolve()
    WORK_DIR = str(ROOT / "work_yolo")
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / "cache" / "ultralytics")
import re, csv, json, time, shutil, hashlib, datetime, getpass
import numpy as np, cv2, yaml, torch
import onnx, onnxruntime as ort
from ultralytics import YOLO
from ultralytics.utils.torch_utils import get_flops

DEVICE = 0 if torch.cuda.is_available() and not FAKE_DATA else "cpu"
KIND = "fake" if FAKE_DATA else "real"
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = ROOT / "yolo" / KIND / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
WEIGHTS_DIR = ROOT / "yolo" / "weights" / (DATA_TAG + ("_fake" if FAKE_DATA else ""))
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_CSV = ROOT / "compare" / ("yolo_compare.csv" if not FAKE_DATA else "yolo_compare_fake.csv")
COMPARE_CSV.parent.mkdir(parents=True, exist_ok=True)
WORK = Path(WORK_DIR); WORK.mkdir(parents=True, exist_ok=True)
NC = len(CLASS_NAMES); CID = {n: i for i, n in enumerate(CLASS_NAMES)}
if FAKE_DATA:   # cỡ nhỏ để chạy CPU nhanh, vẫn giữ quan hệ vuông / chữ nhật
    EPOCHS, BATCH, SEEDS, VARIANTS = 1, 4, SEEDS[:2], list(VARIANT_SPECS)
    VARIANT_SPECS = {k: dict(v, train=96, hw=(64, 96) if v["hw"][0] != v["hw"][1] else (96, 96)) for k, v in VARIANT_SPECS.items()}
print("Thiết bị:", DEVICE, "| Kết quả:", OUT)

def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def write_csv(path, rows, fields):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)

def append_csv(path, rows):
    path = Path(path); old = []
    if path.exists():
        with path.open(encoding="utf-8-sig") as f: old = list(csv.DictReader(f))
    write_csv(path, old + rows, list(dict.fromkeys([k for r in old + rows for k in r])))

def done_keys(path):
    if not Path(path).exists(): return set()
    with Path(path).open(encoding="utf-8-sig") as f: return {r["run_key"] for r in csv.DictReader(f)}

def rf_project(name):
    from roboflow import Roboflow
    key = None
    if IN_COLAB:
        try:
            from google.colab import userdata; key = userdata.get("ROBOFLOW_API_KEY")
        except Exception: pass
    key = key or getpass.getpass("Khoá Roboflow (không lưu): ")
    rf = Roboflow(api_key=key)
    return (rf.workspace(RF_WORKSPACE) if RF_WORKSPACE else rf.workspace()).project(name)

RF_SUFFIX = re.compile(r"_(?:jpg|jpeg|png)\.rf\.[A-Za-z0-9]+$", re.I)
NAME_RE = re.compile(r"car6_p(\d{4})_(\d{4})")
def original_stem(name):
    stem = RF_SUFFIX.sub("", Path(name).stem)
    if not NAME_RE.fullmatch(stem): raise ValueError("Tên không theo mẫu car6_pNNNN_MMMM: " + name)
    return stem

def letterbox(bgr, hw):
    """Thu nhỏ giữ tỉ lệ bằng INTER_AREA, đệm 114, căn giữa. Trả về ảnh, tỉ lệ, (đệm x, đệm y)."""
    h0, w0 = bgr.shape[:2]; H, W = hw; r = min(H / h0, W / w0)
    nw, nh = round(w0 * r), round(h0 * r)
    img = cv2.resize(bgr, (nw, nh), interpolation=cv2.INTER_AREA) if (nw, nh) != (w0, h0) else bgr
    px, py = (W - nw) / 2, (H - nh) / 2
    top, left = int(round(py - 0.1)), int(round(px - 0.1))
    out = cv2.copyMakeBorder(img, top, H - nh - top, left, W - nw - left, cv2.BORDER_CONSTANT, value=(114, 114, 114))
    return out, r, (left, top)

## Dữ liệu
Thật: tải version `RF_VERSION` của `car6_det` dạng `yolov8`, gộp train/valid/test của Roboflow rồi chia lại theo `split_v0.csv` (đặt `Vong3/data/split_v0.csv`). Nhãn đổi số hiệu lớp theo tên; tên lớp lạ làm notebook dừng. Ảnh nền (không có vật) phải có file nhãn rỗng. Giả: tạo ảnh 1280×720 có các khối màu theo lớp.

In [ ]:
def fake_dataset(folder, n=60):
    rng = np.random.default_rng(0); (folder / "images").mkdir(parents=True); (folder / "labels").mkdir()
    colors = [(40, 40, 230), (40, 220, 40), (60, 60, 60), (230, 120, 20), (200, 160, 20), (30, 30, 200),
              (60, 30, 180), (20, 20, 250), (0, 140, 255)]
    rows = []
    for i in range(n):
        p = 10 + i // 6; stem = "car6_p{:04d}_{:04d}".format(p, i % 6 + 1)
        img = np.full((720, 1280, 3), 110, np.uint8); lab = []
        for c in rng.choice(NC, size=int(rng.integers(0, 3)), replace=False):
            w, h = int(rng.integers(120, 260)), int(rng.integers(120, 260))
            x, y = int(rng.integers(0, 1280 - w)), int(rng.integers(0, 720 - h))
            cv2.rectangle(img, (x, y), (x + w, y + h), colors[c], -1)
            if c in (3, 4): cv2.arrowedLine(img, (x + w // 2, y + h // 2), (x + (w // 6 if c == 3 else 5 * w // 6), y + h // 2), (255, 255, 255), 12)
            lab.append("{} {:.6f} {:.6f} {:.6f} {:.6f}".format(c, (x + w / 2) / 1280, (y + h / 2) / 720, w / 1280, h / 720))
        cv2.imwrite(str(folder / "images" / (stem + ".jpg")), img)
        (folder / "labels" / (stem + ".txt")).write_text("\n".join(lab) + ("\n" if lab else ""), encoding="utf-8")
        rows.append([stem + ".jpg", p, "val" if p in (11, 15) else "train"])
    with (folder / "split.csv").open("w", newline="") as f:
        w = csv.writer(f); w.writerow(["file", "pass", "split"]); w.writerows(rows)
    (folder / "data.yaml").write_text(yaml.safe_dump({"names": CLASS_NAMES, "nc": NC}), encoding="utf-8")

if FAKE_DATA:
    RAW = WORK / ("fake_" + RUN_ID); fake_dataset(RAW); split_path = RAW / "split.csv"
elif MODE != "prelabel":
    RAW = WORK / ("rf_" + DATA_TAG)
    if not (RAW / "data.yaml").exists():
        rf_project(RF_PROJECT).version(RF_VERSION).download("yolov8", location=str(RAW), overwrite=True)
    split_path = ROOT / SPLIT_CSV

RECORDS = []
if MODE != "prelabel":
    with open(split_path, encoding="utf-8-sig") as f:
        SPLIT = {Path(r["file"]).stem: (int(r["pass"]), r["split"]) for r in csv.DictReader(f)}
    names = yaml.safe_load((RAW / "data.yaml").read_text(encoding="utf-8"))["names"]
    names = [names[i] for i in sorted(names)] if isinstance(names, dict) else list(names)
    unknown = set(names) - set(CLASS_NAMES)
    if unknown: raise ValueError("Lớp lạ trong export Roboflow: " + ", ".join(sorted(unknown)))
    remap = {i: CID[n] for i, n in enumerate(names)}
    seen = set()
    for p in sorted(RAW.rglob("*")):
        if p.suffix.lower() not in (".jpg", ".jpeg", ".png") or p.parent.name != "images": continue
        stem = original_stem(p.name)
        if stem in seen: raise ValueError("Ảnh trùng tên gốc: " + stem)
        if stem not in SPLIT: raise ValueError("Ảnh không có trong split_v0.csv: " + stem)
        seen.add(stem)
        lp = p.parent.parent / "labels" / (p.stem + ".txt")
        if not lp.exists(): raise ValueError("Thiếu file nhãn: " + lp.name)
        boxes = []
        for line in lp.read_text(encoding="utf-8").split("\n"):
            if not line.strip(): continue
            v = [float(t) for t in line.split()]
            if len(v) != 5: raise ValueError("Nhãn phải là hộp 5 số (không dùng đa giác): " + lp.name)
            c, x, y, w, h = v
            if not 0 < w <= 1 or not 0 < h <= 1: raise ValueError("Hộp sai kích thước: " + lp.name)
            boxes.append([remap[int(c)], x, y, w, h])
        RECORDS.append(dict(stem=stem, image=p, boxes=np.array(boxes, np.float32).reshape(-1, 5),
                            pass_id=SPLIT[stem][0], split=SPLIT[stem][1]))
    TRAIN = [r for r in RECORDS if r["split"] == "train"]; VAL = [r for r in RECORDS if r["split"] == "val"]
    assert TRAIN and VAL and not {r["pass_id"] for r in TRAIN} & {r["pass_id"] for r in VAL}
    print("Ảnh: train", len(TRAIN), "/ val", len(VAL))
    for c, n in enumerate(CLASS_NAMES):
        cnt = [sum(int((r["boxes"][:, 0] == c).sum()) for r in part) for part in (TRAIN, VAL)]
        print("  {:14s} hộp train/val: {}".format(n, cnt) + ("  <- thiếu ở một tập" if min(cnt) == 0 else ""))
    write_csv(OUT / "split_used.csv", [{k: r[k] for k in ("stem", "pass_id", "split")} | {"boxes": len(r["boxes"])} for r in RECORDS],
              ["stem", "pass_id", "split", "boxes"])

## Tăng cường và dựng tập train/val theo cỡ
Với mỗi cỡ train (cạnh dài 320/416/640) dựng một thư mục riêng: ảnh thu nhỏ trước bằng INTER_AREA (cạnh dài = cỡ train), val chỉ ảnh gốc, train gồm ảnh gốc + `AUG_COPIES` bản đổi màu + bản lật đổi nhãn `left`/`right`. Mọi bản biến thể chỉ sinh từ ảnh train (sau khi chia), không rò sang val. `augmentation_manifest.csv` ghi nguồn từng ảnh.

In [ ]:
def camera_aug(image, rng):
    """Chỉ đổi màu/cường độ, giữ hình học nên hộp nhãn không đổi."""
    x = image.astype(np.float32); h, w = x.shape[:2]
    mean = x.mean(axis=(0, 1), keepdims=True)
    x = (x - mean) * rng.uniform(0.8, 1.2) + mean
    x *= rng.uniform(0.75, 1.25) * rng.uniform(0.85, 1.15, (1, 1, 3))
    if rng.random() < 0.3:
        x0, x1, y0 = int(rng.uniform(0, .35) * w), int(rng.uniform(.65, 1) * w), int(rng.uniform(0, .45) * h)
        x[y0:, x0:x1] *= rng.uniform(0.12, 0.4) * np.array([rng.uniform(.85, 1.2), rng.uniform(.65, 1), rng.uniform(.6, 1)])
    if rng.random() < 0.3:
        k = int(rng.choice([3, 5, 7])); kern = np.zeros((k, k), np.float32)
        if rng.random() < 0.5: kern[k // 2, :] = 1 / k
        else: kern[:, k // 2] = 1 / k
        x = cv2.filter2D(x, -1, kern)
    if rng.random() < 0.5: x += rng.normal(0, rng.uniform(1, 8), x.shape)
    return np.clip(x, 0, 255).astype(np.uint8)

def label_text(boxes):
    return "".join("{} {:.6f} {:.6f} {:.6f} {:.6f}\n".format(int(b[0]), *b[1:]) for b in boxes)

def flippable(boxes):
    names = {CLASS_NAMES[int(c)] for c in boxes[:, 0]}
    return bool(names & set(FLIP_SWAP)) and not names & NO_FLIP

def flip_boxes(boxes):
    b = boxes.copy(); b[:, 1] = 1 - b[:, 1]
    b[:, 0] = [CID[FLIP_SWAP.get(CLASS_NAMES[int(c)], CLASS_NAMES[int(c)])] for c in b[:, 0]]
    return b

def build_split(train_size, final=False):
    """Thư mục dataset cho một cỡ train; dựng một lần, dùng lại trong phiên."""
    folder = WORK / "ds_{}_{}{}".format(DATA_TAG, train_size, "_final" if final else "")
    yaml_path = folder / "data.yaml"
    if yaml_path.exists(): return yaml_path
    rng = np.random.default_rng(0); manifest = []
    for part in ("train", "val"):
        for sub in ("images", "labels"): (folder / part / sub).mkdir(parents=True, exist_ok=True)
    for r in RECORDS:
        part = "train" if final or r["split"] == "train" else "val"
        also_val = final and r["split"] == "val"   # FINAL_FIT: giữ thư mục val để Ultralytics không báo thiếu
        src = cv2.imread(str(r["image"]))
        if src is None or src.shape[:2] != (720, 1280): raise ValueError("Ảnh phải 1280×720: " + r["stem"])
        s = train_size / max(src.shape[:2])
        img = cv2.resize(src, (round(src.shape[1] * s), round(src.shape[0] * s)), interpolation=cv2.INTER_AREA)
        items = [(r["stem"], img, r["boxes"])]
        if part == "train":
            items += [("{}_aug{}".format(r["stem"], j), camera_aug(img, rng), r["boxes"]) for j in range(AUG_COPIES)]
            if FLIP_SIGNS and flippable(r["boxes"]): items.append((r["stem"] + "_flip", img[:, ::-1].copy(), flip_boxes(r["boxes"])))
        for name, im, bx in items:
            cv2.imwrite(str(folder / part / "images" / (name + ".jpg")), im, [cv2.IMWRITE_JPEG_QUALITY, 95])
            (folder / part / "labels" / (name + ".txt")).write_text(label_text(bx), encoding="utf-8")
            if also_val and name == r["stem"]:
                cv2.imwrite(str(folder / "val" / "images" / (name + ".jpg")), im, [cv2.IMWRITE_JPEG_QUALITY, 95])
                (folder / "val" / "labels" / (name + ".txt")).write_text(label_text(bx), encoding="utf-8")
            if name != r["stem"]: manifest.append(dict(image=name, source=r["stem"], pass_id=r["pass_id"], split=part))
    write_csv(folder / "augmentation_manifest.csv", manifest, ["image", "source", "pass_id", "split"])
    yaml_path.write_text(yaml.safe_dump({"path": str(folder), "train": "train/images", "val": "val/images",
                                         "nc": NC, "names": CLASS_NAMES}), encoding="utf-8")
    print("Dựng", folder.name, "| bản biến thể:", len(manifest), "| bản lật:", sum(m["image"].endswith("_flip") for m in manifest))
    return yaml_path

## Huấn luyện
Học chuyển giao từ trọng số COCO. Tắt `fliplr`, `mosaic`, `mixup`, `copy_paste`, HSV và mọi biến đổi hình học của Ultralytics (đã có bản biến thể ngoại tuyến). Lấy trọng số **epoch cuối** (`last.pt`), không chọn theo val. Trọng số lưu ở `Vong3/yolo/weights/<DATA_TAG>/<mô hình>_<cỡ>_s<seed>.pt` cùng file `.json` mô tả; nếu đã có thì dùng lại, không train lại.

In [ ]:
def train_or_load(spec, seed, final=False):
    tkey = "{}_{}_s{}{}".format(spec["model"], spec["train"], seed, "_final" if final else "")
    wpath = WEIGHTS_DIR / (tkey + ".pt")
    if wpath.exists():
        print("  Dùng lại trọng số:", wpath.name); return wpath, None
    data_yaml = build_split(spec["train"], final)
    prev = Path.cwd(); t0 = time.time()
    try:
        os.chdir(WORK)   # trọng số COCO tải về nằm ở đĩa Colab
        det = YOLO(spec["model"] + (".yaml" if FAKE_DATA else ".pt"))
        det.train(data=str(data_yaml), imgsz=spec["train"], epochs=EPOCHS, batch=BATCH, device=DEVICE, workers=2,
                  seed=seed, deterministic=False, pretrained=not FAKE_DATA, amp=not FAKE_DATA, cos_lr=True,
                  fliplr=0.0, flipud=0.0, mosaic=0.0, close_mosaic=0, mixup=0.0, copy_paste=0.0,
                  hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, degrees=0.0, translate=0.0, scale=0.0, shear=0.0, perspective=0.0,
                  erasing=0.0, val=not final, plots=False, save=True, cache=False, patience=0,
                  project=str(WORK / "runs"), name=tkey, exist_ok=True, verbose=False)
        last = Path(det.trainer.last)
    finally:
        os.chdir(prev)
    shutil.copy2(last, wpath)
    res_csv = last.parent.parent / "results.csv"
    if res_csv.exists(): shutil.copy2(res_csv, WEIGHTS_DIR / (tkey + "_results.csv"))
    write_json(wpath.with_suffix(".json"), dict(model=spec["model"], train_size=spec["train"], seed=seed, data_tag=DATA_TAG,
               final_fit=final, epochs=EPOCHS, class_names=CLASS_NAMES, aug_copies=AUG_COPIES, flip_signs=FLIP_SIGNS))
    return wpath, round((time.time() - t0) / 60, 1)

## Đánh giá
- **mAP50, mAP50-95** (toàn bộ và từng lớp) bằng `model.val` của Ultralytics ở đúng cỡ suy luận của biến thể (D2, D4 dùng `rect=True` → 256×416, 384×640).
- **Ma trận nhầm lẫn, recall/precision từng lớp** bằng ghép hộp một-một IoU ≥ 0,5, ảnh đã letterbox INTER_AREA như trên xe; quét `CONF_THRESHOLDS` × 5 mức sáng. Mỗi dòng CSV = lần chạy × mức sáng × ngưỡng conf, có cột riêng cho các cặp dễ nhầm `left`↔`no_left`, `right`↔`no_right`.
- Ảnh lỗi ở `EVAL_CONF`, độ sáng 1,0: `<run>_errors.csv`. Màu đèn sẽ được kiểm lại bằng HSV và bỏ phiếu nhiều khung trong C++, không đo ở đây.

In [ ]:
def xywh_to_xyxy(a):
    a = np.asarray(a, float).reshape(-1, 4); return np.concatenate([a[:, :2] - a[:, 2:] / 2, a[:, :2] + a[:, 2:] / 2], 1)

def pair_iou(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    inter = np.clip(rb - lt, 0, None).prod(2)
    area = lambda z: np.clip(z[:, 2:] - z[:, :2], 0, None).prod(1)
    return inter / (area(a)[:, None] + area(b)[None] - inter + 1e-9)

def confusion_one(gt, pred):
    """gt: [lớp,x1,y1,x2,y2]; pred: [lớp,conf,x1,y1,x2,y2]; hàng = thật, cột = dự đoán, chỉ số NC = nền."""
    mat = np.zeros((NC + 1, NC + 1), int); ug, up = set(), set()
    if len(gt) and len(pred):
        iou = pair_iou(gt[:, 1:], pred[:, 2:])
        for g, p in sorted(np.argwhere(iou >= MATCH_IOU), key=lambda z: -iou[z[0], z[1]]):
            if g in ug or p in up: continue
            ug.add(g); up.add(p); mat[int(gt[g, 0]), int(pred[p, 0])] += 1
    for g in set(range(len(gt))) - ug: mat[int(gt[g, 0]), NC] += 1
    for p in set(range(len(pred))) - up: mat[NC, int(pred[p, 0])] += 1
    return mat

def gt_in_letterbox(boxes, hw):
    """Hộp chuẩn hoá theo ảnh 1280×720 -> xyxy chuẩn hoá theo ảnh letterbox."""
    _, r, (px, py) = letterbox(np.zeros((720, 1280, 3), np.uint8), hw)
    xy = xywh_to_xyxy(boxes[:, 1:]) * [1280, 720, 1280, 720] * r + [px, py, px, py]
    return np.column_stack([boxes[:, 0], xy / [hw[1], hw[0], hw[1], hw[0]]]).reshape(-1, 5)

def predict_lb(det, images, hw, conf):
    """Dự đoán trên ảnh đã letterbox đúng cỡ (Ultralytics không thu nhỏ thêm)."""
    out = []
    for i in range(0, len(images), 32):
        for res in det.predict(images[i:i + 32], imgsz=list(hw), conf=conf, iou=NMS_IOU, device=DEVICE, verbose=False, rect=False):
            b = res.boxes
            out.append(np.column_stack([b.cls.cpu().numpy(), b.conf.cpu().numpy(), b.xyxyn.cpu().numpy()]).reshape(-1, 6))
    return out

def evaluate(det, spec, data_yaml, run_name):
    hw = spec["hw"]
    m = det.val(data=str(data_yaml), imgsz=max(hw), rect=hw[0] != hw[1], batch=16, conf=0.001, iou=0.6,
                device=DEVICE, plots=False, verbose=False, split="val")
    run = dict(map50=float(m.box.map50), map50_95=float(m.box.map))
    for k, c in enumerate(m.box.ap_class_index):
        run["map50_" + CLASS_NAMES[int(c)]] = float(m.box.ap50[k]); run["map50_95_" + CLASS_NAMES[int(c)]] = float(m.box.ap[k])
    lb = [letterbox(cv2.imread(str(r["image"])), hw)[0] for r in VAL]
    gts = [gt_in_letterbox(r["boxes"], hw) for r in VAL]
    rows, errors = [], []
    for gain in BRIGHTNESS:
        imgs = [np.clip(im.astype(np.float32) * gain, 0, 255).astype(np.uint8) for im in lb]
        preds = predict_lb(det, imgs, hw, min(CONF_THRESHOLDS + [EVAL_CONF]))
        for conf in sorted(set(CONF_THRESHOLDS + [EVAL_CONF])):
            total = np.zeros((NC + 1, NC + 1), int)
            for r, g, p in zip(VAL, gts, preds):
                mat = confusion_one(g, p[p[:, 1] >= conf]); total += mat
                if gain == 1.0 and conf == EVAL_CONF and mat.sum() - np.trace(mat):
                    errors.append(dict(stem=r["stem"], pass_id=r["pass_id"], errors=int(mat.sum() - np.trace(mat))))
            row = dict(run, brightness=gain, conf=conf)
            tp = np.trace(total[:NC, :NC]); row["tp"] = int(tp); row["fp"] = int(total[:, :NC].sum() - tp); row["fn"] = int(total[:NC].sum() - tp)
            row["precision"] = tp / max(1, total[:, :NC].sum()); row["recall"] = tp / max(1, total[:NC].sum())
            for c, n in enumerate(CLASS_NAMES):
                act, pre = total[c].sum(), total[:, c].sum()
                row["recall_" + n] = total[c, c] / act if act else None; row["precision_" + n] = total[c, c] / pre if pre else None
            for a, b in (("left", "no_left"), ("no_left", "left"), ("right", "no_right"), ("no_right", "right"), ("red", "green"), ("green", "red")):
                row["confuse_{}_as_{}".format(a, b)] = int(total[CID[a], CID[b]])
            rows.append(row)
            if gain == 1.0 and conf == EVAL_CONF: save_confusion(total, OUT / (run_name + "_confusion"))
    write_csv(OUT / (run_name + "_errors.csv"), sorted(errors, key=lambda e: -e["errors"]), ["stem", "pass_id", "errors"])
    return rows

def save_confusion(mat, base):
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    labels = CLASS_NAMES + ["nen"]
    with open(str(base) + ".csv", "w", encoding="utf-8-sig", newline="") as f:
        w = csv.writer(f); w.writerow(["that\\du_doan"] + labels); [w.writerow([n] + list(r)) for n, r in zip(labels, mat.tolist())]
    fig, ax = plt.subplots(figsize=(8, 7)); ax.imshow(mat, cmap="Blues")
    ax.set(xticks=range(NC + 1), yticks=range(NC + 1), xticklabels=labels, yticklabels=labels, xlabel="Dự đoán", ylabel="Nhãn thật")
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    for (i, j), v in np.ndenumerate(mat): ax.text(j, i, str(v), ha="center", va="center", fontsize=8)
    fig.tight_layout(); fig.savefig(str(base) + ".png"); plt.close(fig)

## Xuất ONNX và đối chiếu
ONNX opset 12, đầu vào cố định `[1,3,H,W]` theo cỡ suy luận của biến thể, FP32, simplify, không NMS. Đầu ra `[1,4+9,N]`: `cx,cy,w,h` theo pixel ảnh letterbox và xác suất từng lớp. Đối chiếu onnxruntime với PyTorch trên ≥ 20 ảnh thật đã letterbox INTER_AREA: sai số điểm lớp < 0,001, sai số toạ độ < 0,01 px; lưu tensor vào/ra để kiểm TensorRT trên xe. `_info.json` ghi sẵn tỉ lệ và đệm cho ảnh 1280×720 để C++ đổi hộp về ảnh gốc.

In [ ]:
def export_onnx(wpath, spec, name, extra):
    from ultralytics import YOLO as _Y
    H, W = spec["hw"]; det = _Y(str(wpath))
    exported = Path(det.export(format="onnx", imgsz=[H, W], batch=1, opset=12, simplify=True, dynamic=False,
                               half=False, nms=False, device="cpu"))
    path = OUT / (name + ".onnx"); shutil.copy2(exported, path)
    g = onnx.load(str(path)); onnx.checker.check_model(g)
    assert next(o.version for o in g.opset_import if o.domain in ("", "ai.onnx")) == 12
    assert [d.dim_value for d in g.graph.input[0].type.tensor_type.shape.dim] == [1, 3, H, W]
    sess = ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])
    pt = _Y(str(wpath)).model.cpu().float().eval()
    recs = [r for r in RECORDS if r["split"] == "val"] + [r for r in RECORDS if r["split"] == "train"]
    if len(recs) < 20 and not FAKE_DATA: raise ValueError("Cần ít nhất 20 ảnh thật để đối chiếu ONNX")
    ref = OUT / (name + "_onnx_reference"); ref.mkdir(); checks = []
    for i, r in enumerate(recs[:20]):
        im = letterbox(cv2.imread(str(r["image"])), (H, W))[0]
        x = np.ascontiguousarray(im[:, :, ::-1].transpose(2, 0, 1)[None], dtype=np.float32) / 255
        with torch.no_grad():
            a = pt(torch.from_numpy(x)); a = (a[0] if isinstance(a, (tuple, list)) else a).numpy()
        b = sess.run(None, {sess.get_inputs()[0].name: x})[0]
        assert a.shape == b.shape and b.shape[1] == 4 + NC
        se, be = float(np.abs(a[:, 4:] - b[:, 4:]).max()), float(np.abs(a[:, :4] - b[:, :4]).max())
        assert se < 1e-3 and be < 1e-2, "ONNX lệch PyTorch"
        np.savez_compressed(ref / "{:03d}.npz".format(i), images=x, output=a)
        checks.append(dict(stem=r["stem"], max_score_error=se, max_box_error_px=be))
    write_csv(OUT / (name + "_onnx_check.csv"), checks, ["stem", "max_score_error", "max_box_error_px"])
    _, r0, (px, py) = letterbox(np.zeros((720, 1280, 3), np.uint8), (H, W))
    import ultralytics
    info = {"task": "detection", "architecture": spec["model"], "class_names": CLASS_NAMES, "num_classes": NC,
            "source_image_hw": [720, 1280], "input_name": sess.get_inputs()[0].name, "input_shape": [1, 3, H, W],
            "input_hw": [H, W], "input_dtype": "float32", "color_order": "RGB", "layout": "NCHW",
            "normalization": {"scale": 1 / 255, "mean": [0, 0, 0], "std": [1, 1, 1]}, "preprocessing_in_model": False,
            "resize": "letterbox: cv2.resize INTER_AREA giữ tỉ lệ, đệm 114, căn giữa",
            "letterbox_1280x720": {"scale": r0, "resized_wh": [round(1280 * r0), round(720 * r0)], "pad_left": px, "pad_top": py},
            "output_name": sess.get_outputs()[0].name, "output_shape": list(b.shape),
            "output_type": "cxcywh_pixels_and_class_probabilities", "objectness": False,
            "nms_in_model": False, "nms_class_aware": True, "conf_threshold": EVAL_CONF, "nms_iou": NMS_IOU,
            "opset": 12, "dynamic": False, "latency_measured": False, "onnx_check_images": len(checks),
            "onnx_max_score_error": max(c["max_score_error"] for c in checks),
            "onnx_max_box_error_px": max(c["max_box_error_px"] for c in checks), "fake_data": FAKE_DATA,
            "ultralytics_version": ultralytics.__version__, "torch_version": torch.__version__, "onnxruntime_version": ort.__version__}
    info.update(extra); write_json(OUT / (name + "_info.json"), info)
    print("  ONNX:", path)

## Chạy so sánh hoặc FINAL_FIT
`run_key = DATA_TAG|biến thể|seed`. Lần chạy đã có trong `yolo_compare.csv` được bỏ qua.

In [ ]:
SUMMARY = []
if MODE in ("compare", "final"):
    names = VARIANTS if MODE == "compare" else [FINAL_VARIANT]
    seeds = SEEDS if MODE == "compare" else [FINAL_SEED]
    skip = done_keys(COMPARE_CSV) if MODE == "compare" else set()
    for v in names:
        spec = VARIANT_SPECS[v]
        for k, seed in enumerate(seeds):
            key = "{}|{}|{}".format(DATA_TAG, v, seed)
            if key in skip: print("Bỏ qua (đã có):", key); continue
            print(v, spec, "seed", seed)
            wpath, minutes = train_or_load(spec, seed, final=MODE == "final")
            det = YOLO(str(wpath)); name = "yolo_{}_s{}{}".format(v, seed, "_final" if MODE == "final" else "")
            assert [det.names[i] for i in range(NC)] == CLASS_NAMES, "Thứ tự lớp của trọng số không khớp"
            params = int(sum(p.numel() for p in det.model.parameters())); gflops = float(get_flops(det.model, imgsz=list(spec["hw"])))
            if MODE == "compare":
                base = dict(run_key=key, run_id=RUN_ID, data_tag=DATA_TAG, variant=v, model=spec["model"], train_size=spec["train"],
                            input_h=spec["hw"][0], input_w=spec["hw"][1], seed=seed, epochs=EPOCHS, aug_copies=AUG_COPIES,
                            flip_signs=FLIP_SIGNS, params=params, gflops=round(gflops, 4), train_min=minutes,
                            n_train=len(TRAIN), n_val=len(VAL), weights=str(wpath.relative_to(ROOT)))
                rows = [dict(base, **r) for r in evaluate(det, spec, build_split(spec["train"]), name)]
                append_csv(COMPARE_CSV, rows); SUMMARY += rows
                r1 = next(r for r in rows if r["brightness"] == 1.0 and r["conf"] == EVAL_CONF)
                print("  mAP50 {:.3f} | mAP50-95 {:.3f} | recall {:.3f} | precision {:.3f}".format(r1["map50"], r1["map50_95"], r1["recall"], r1["precision"]))
            if MODE == "final" or (EXPORT_FIRST_SEED and k == 0):
                export_onnx(wpath, spec, name, dict(variant=v, seed=seed, data_tag=DATA_TAG, final_fit=MODE == "final",
                            trained_on="train+val" if MODE == "final" else "train", parameters=params, gflops=gflops))
    print("Xong. Bảng so sánh:", COMPARE_CSV if MODE == "compare" else "(FINAL_FIT không đánh giá)")

## Tóm tắt so sánh
Trung bình ± độ lệch giữa các seed ở `EVAL_CONF`, độ sáng 1,0 và trung bình 5 mức sáng.

In [ ]:
if MODE == "compare" and COMPARE_CSV.exists():
    import pandas as pd
    try: from IPython.display import display
    except ImportError: display = print
    df = pd.read_csv(COMPARE_CSV, encoding="utf-8-sig"); df = df[(df["data_tag"] == DATA_TAG) & (df["conf"] == EVAL_CONF)]
    cols = [c for c in ["map50", "map50_95", "recall", "precision", "recall_red", "recall_green", "recall_left", "recall_right",
                        "recall_no_left", "recall_no_right", "confuse_left_as_no_left", "confuse_no_left_as_left",
                        "confuse_right_as_no_right", "confuse_no_right_as_right"] if c in df]
    for title, part in (("Độ sáng 1,0", df[df["brightness"] == 1.0]), ("Trung bình 5 mức sáng", df)):
        print(title); display(part.groupby(["variant", "seed"])[cols].mean().groupby("variant").agg(["mean", "std"]).round(3))
    print(df.groupby("variant")[["input_h", "input_w", "params", "gflops", "train_min"]].first())

## Gán nhãn sơ bộ cho `car6_det`
Đặt `MODE="prelabel"`, `FAKE_DATA=False`, `PRELABEL_WEIGHTS` trỏ tới `.pt` trong `Vong3/yolo/weights/...`, chọn biến thể có cùng cỡ train (mặc định đọc từ `.json` cạnh trọng số: suy luận vuông theo cỡ train). Notebook tìm ảnh chưa vào Dataset, letterbox INTER_AREA, dự đoán, đổi hộp về ảnh gốc, gửi `save_annotation(is_prediction=True, annotation_overwrite=False, add_to_dataset=False)`. Không ghi đè nhãn đã có, không tự đưa ảnh vào Dataset. Nhật ký mã ảnh đã gửi: `yolo/prelabel_done.json`. Chạy `PRELABEL_LIMIT=20` trước, kiểm trên Roboflow rồi mới đặt `None`.

In [ ]:
if MODE == "prelabel" or FAKE_DATA:
    import requests, inspect
    if FAKE_DATA:
        wp = Path(sorted(WEIGHTS_DIR.glob("*.pt"))[0]); meta = json.loads(wp.with_suffix(".json").read_text(encoding="utf-8"))
        pending = [dict(id=r["stem"], name=r["stem"] + ".jpg", local=r["image"]) for r in VAL[:2]]
    else:
        wp = Path(PRELABEL_WEIGHTS).expanduser().resolve()
        if not wp.is_file() or ROOT not in wp.parents: raise ValueError("PRELABEL_WEIGHTS phải là file .pt trong Vong3")
        meta = json.loads(wp.with_suffix(".json").read_text(encoding="utf-8"))
        project = rf_project(RF_PROJECT); pending = []; offset = 0
        while True:
            page = project.search(in_dataset=False, offset=offset, limit=250, fields=["id", "name"])
            if not isinstance(page, list): raise RuntimeError("Định dạng search Roboflow thay đổi")
            pending += page; offset += 250
            if len(page) < 250: break
    det = YOLO(str(wp)); assert [det.names[i] for i in range(NC)] == CLASS_NAMES
    hw = (meta["train_size"], meta["train_size"])
    journal = ROOT / "yolo" / "prelabel_done.json"
    done = json.loads(journal.read_text(encoding="utf-8")) if journal.exists() and not FAKE_DATA else []
    prefix = "{}/{}:".format(RF_WORKSPACE or "default", RF_PROJECT)
    pending = [p for p in pending if prefix + str(p["id"]) not in set(done)]
    if PRELABEL_LIMIT is not None: pending = pending[:PRELABEL_LIMIT]
    PRE = OUT / "prelabel"; PRE.mkdir(exist_ok=True); logs = []
    for item in pending:
        if FAKE_DATA: bgr = cv2.imread(str(item["local"]))
        else:
            data = project.image(item["id"]); urls = data.get("urls") or {}
            url = urls.get("original") or data.get("image_url")
            if not url: raise RuntimeError("Roboflow không trả URL ảnh")
            resp = requests.get(url, timeout=(10, 30)); resp.raise_for_status()
            bgr = cv2.imdecode(np.frombuffer(resp.content, np.uint8), cv2.IMREAD_COLOR)
        if bgr is None: raise RuntimeError("Ảnh không đọc được: " + str(item["name"]))
        im, r, (px, py) = letterbox(bgr, hw); h0, w0 = bgr.shape[:2]
        p = predict_lb(det, [im], hw, PRELABEL_CONF)[0]
        rows = []
        for c, _, x1, y1, x2, y2 in p:   # xyxy chuẩn hoá theo letterbox -> xywh chuẩn hoá theo ảnh gốc
            X1, X2 = (np.array([x1, x2]) * hw[1] - px) / r / w0; Y1, Y2 = (np.array([y1, y2]) * hw[0] - py) / r / h0
            X1, X2, Y1, Y2 = np.clip([X1, X2, Y1, Y2], 0, 1)
            if X2 > X1 and Y2 > Y1: rows.append("{} {:.6f} {:.6f} {:.6f} {:.6f}".format(int(c), (X1 + X2) / 2, (Y1 + Y2) / 2, X2 - X1, Y2 - Y1))
        label = PRE / (hashlib.sha256(str(item["id"]).encode()).hexdigest()[:20] + ".txt")
        label.write_text("\n".join(rows) + ("\n" if rows else ""), encoding="utf-8"); status = "khong_thay_vat"
        if rows:
            if FAKE_DATA: status = "gia_lap"
            else:
                kw = dict(annotation_path=str(label), annotation_labelmap=dict(enumerate(CLASS_NAMES)), image_id=item["id"],
                          is_prediction=True, annotation_overwrite=False)
                if "add_to_dataset" in inspect.signature(project.save_annotation).parameters: kw["add_to_dataset"] = False
                project.save_annotation(**kw)
                status = "da_gui_can_duyet"; done.append(prefix + str(item["id"])); write_json(journal, sorted(set(done)))
        logs.append(dict(image_id=item["id"], name=item["name"], boxes=len(rows), status=status))
        write_csv(PRE / "prelabel_log.csv", logs, ["image_id", "name", "boxes", "status"])
    print("Ảnh xử lý:", len(logs), "| Mở Roboflow để duyệt.")